<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>CLIP 평가(CLIP Evaluation)</strong>에 대해 학습합니다.</br></br>
>CLIP 모델로 생성 이미지와 텍스트 간의 유사도를 학습해봅시다.

</br>

# CLIP (Contrastive Language-Image Pre-training)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">텍스트와 이미지를 동일한 임베딩 공간</mark>에 매핑하여 유사도를 계산하는 모델입니다.

기존 비전 모델은 사전에 정의된 카테고리만 분류할 수 있어 새 카테고리를 추가하려면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">재학습</mark>이 필요합니다. CLIP은 이미지와 텍스트를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">동일한 임베딩 공간에 매핑</mark>하여 두 모달리티 간 코사인 유사도를 직접 비교함으로써 이 한계를 극복합니다. 분류하고 싶은 카테고리를 텍스트로 서술하기만 하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">별도의 학습 없이</mark> 이미지가 어떤 텍스트 설명과 가장 유사한지 계산하여 분류할 수 있으며, 새로운 카테고리도 텍스트 작성만으로 즉시 대응 가능하다는 점이 제로샷 분류의 핵심 혁신입니다.</br>이 장을 학습하기 위해서는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">임 베딩</mark>(고차원 데이터를 고정 크기의 밀집 벡터로 변환한 표현), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">코사인 유사도</mark>(두 벡터 사이의 각도로 유사도를 측정하며 1에 가까울수록 유사), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Contrastive Learning</mark>(같은 쌍은 가깝게, 다른 쌍은 멀어지도록 학습하는 자기지도 기법)에 대한 이해가 도움이 됩니다.

</br>

## CLIPProcessor & CLIPModel

In [1]:
# TODO 1: CLIP 모델과 프로세서를 사전학습 모델로부터 로드하고, 이미지와 여러 텍스트 설명을 전처리하여 입력을 준비해봅시다.

from transformers import CLIPProcessor, CLIPModel
from PIL import Image

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print(f"모델 파라미터: {sum(p.numel() for p in model.parameters()):,}")

# 이미지와 텍스트 전처리
image = Image.open("cat.jpg")
inputs = processor(
    text=["a cat", "a dog", "a car"],
    images=image,
    return_tensors="pt",
    padding=True
)
print(f"pixel_values: {inputs['pixel_values'].shape}")
print(f"input_ids: {inputs['input_ids'].shape}")

config.json: 0.00B [00:00, ?B/s]

c:\Users\SSAFY\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SSAFY\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Fall

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

모델 파라미터: 151,277,313


FileNotFoundError: [Errno 2] No such file or directory: 'cat.jpg'

</br>

## 유사도 계산

In [ ]:
# TODO 2: 모델로 추론한 뒤, 이미지-텍스트 유사도 스코어에 소프트맥스를 적용하여 각 텍스트별 유사도 확률을 출력해봅시다.

import torch

with torch.no_grad():
    outputs = model(**inputs)

# logits_per_image: 이미지와 각 텍스트의 유사도
logits_per_image = outputs.logits_per_image  # (1, 3)
print(f"유사도 스코어: {logits_per_image.data.round(decimals=2)}")

# 확률로 변환
probs = logits_per_image.softmax(dim=1)
print(f"확률: {probs.data.round(decimals=3)}")

labels = ["a cat", "a dog", "a car"]
for label, prob in zip(labels, probs[0]):
    print(f"  {label}: {prob.item():.1%}")

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">텍스트</th>
      <th style="text-align:center">유사도 스코어</th>
      <th style="text-align:center">확률</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">a cat</td><td style="text-align:center">24.56</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">95.2%</mark></td></tr>
    <tr><td style="text-align:center">a dog</td><td style="text-align:center">21.23</td><td style="text-align:center">3.4%</td></tr>
    <tr><td style="text-align:center">a car</td><td style="text-align:center">18.45</td><td style="text-align:center">1.4%</td></tr>
  </tbody>
</table>

💡logits_per_image vs logits_per_text
> `logits_per_image`: 하나의 이미지 vs 여러 텍스트 (이미지 기준)</br>
> `logits_per_text`: 하나의 텍스트 vs 여러 이미지 (텍스트 기준)

</br>

## Zero-shot 분류
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">라벨 학습 없이</mark> 텍스트 설명만으로 이미지를 분류합니다.

In [ ]:
# TODO 3: 여러 텍스트 라벨을 정의하고, 이미지와 함께 전처리 후 모델로 추론하여 가장 유사도가 높은 라벨을 Zero-shot 분류 결과로 출력해봅시다.

labels = ["a photo of a cat", "a photo of a dog", "a photo of a bird"]

inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)
outputs = model(**inputs)

probs = outputs.logits_per_image.softmax(dim=1)
predicted = labels[probs.argmax()]
print(f"예측: {predicted}")
print(f"확률: {probs.data.round(decimals=3)}")

</br>

## 생성 이미지 품질 평가
> 생성된 이미지가 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">원본 프롬프트와 얼마나 일치</mark>하는지 CLIP Score로 평가합니다.

In [ ]:
# TODO 4: 이미지와 텍스트를 입력받아 CLIP 유사도 점수를 반환하는 함수를 정의하고, 생성된 이미지와 프롬프트 간의 CLIP Score를 계산해봅시다.

def clip_score(image, text, model, processor):
    inputs = processor(text=[text], images=image, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits_per_image.item()

score = clip_score(image, "a cat sitting on a windowsill", model, processor)
print(f"CLIP Score: {score:.2f}")

💡CLIP Score가 높으면 좋은 이미지인가?
> CLIP Score는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">텍스트-이미지 정합도</mark>만 측정합니다.</br>
> 이미지 품질(해상도, 아티팩트)은 별도의 지표(FID, IS)로 평가해야 합니다.